In [4]:
import sys
sys.path.append("../src")

import config
print(dir(config))

from config import EICU_DB
print(EICU_DB)

import os
os.path.getsize(EICU_DB)

['BASE_DIR', 'DATA_DIR', 'EICU_DB', 'Path', 'RAW_DATA_DIR', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__']
/Users/jackanderson/Downloads/clinical-mortality-ml/data/raw/eicu_v2_0_1.sqlite3


296071168

In [6]:
import sys
sys.path.append("../src")

from database import run_query

df = run_query("SELECT name FROM sqlite_master WHERE type='table'")
df

,name
0,apacheapsvar
1,apachepatientresult
2,apachepredvar
3,careplancareprovider
4,careplaneol
5,careplangeneral
6,careplangoal
7,careplaninfectiousdisease
8,diagnosis
9,hospital


In [8]:
patients = run_query("SELECT * FROM patient")
patients

,patientunitstayid,patienthealthsystemstayid,gender,age,ethnicity,hospitalid,wardid,apacheadmissiondx,admissionheight,hospitaladmittime24,...,unitadmitsource,unitvisitnumber,unitstaytype,admissionweight,dischargeweight,unitdischargetime24,unitdischargeoffset,unitdischargelocation,unitdischargestatus,uniquepid
0,141764,129391,Female,87,Caucasian,59,91,,157.5,23:36:00,...,ICU to SDU,2,stepdown/other,,,18:58:00,344,Home,Alive,002-1039
1,141765,129391,Female,87,Caucasian,59,91,"Rhythm disturbance (atrial, supraventricular)",157.5,23:36:00,...,Emergency Department,1,admit,46.5,45,13:14:00,2250,Step-Down Unit (SDU),Alive,002-1039
2,143870,131022,Male,76,Caucasian,68,103,"Endarterectomy, carotid",167,20:46:00,...,Operating Room,1,admit,77.5,79.4,10:00:00,793,Floor,Alive,002-12289
3,144815,131736,Female,34,Caucasian,56,82,"Overdose, other toxin, poison or drug",172.7,01:44:00,...,Emergency Department,1,admit,60.3,60.7,20:48:00,1121,Other External,Alive,002-1116
4,145427,132209,Male,61,Caucasian,68,103,"GI perforation/rupture, surgery for",177.8,23:48:00,...,Operating Room,1,admit,91.7,93.1,22:47:00,1369,Floor,Alive,002-12243
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2515,3351763,2741766,Female,62,Caucasian,459,1108,"Fistula/abscess, surgery for (not inflammatory...",165.1,16:08:00,...,Operating Room,1,admit,134.5,133.3,19:24:00,5394,Step-Down Unit (SDU),Alive,035-10391
2516,3352230,2742186,Male,41,African American,458,1107,"CABG alone, coronary artery bypass grafting",177.8,21:21:00,...,Operating Room,2,transfer,127,128.5,21:34:00,4261,Telemetry,Alive,035-10089
2517,3352231,2742186,Male,41,African American,458,1104,"Infarction, acute myocardial (MI)",177.8,21:21:00,...,Direct Admit,1,admit,127,135.2,22:26:00,1369,Other ICU,Alive,035-10089
2518,3352333,2742269,Male,72,Caucasian,458,1111,GI obstruction,177.8,20:00:00,...,Other Hospital,1,admit,68.3,66.5,17:26:00,4166,Floor,Alive,035-10041


In [10]:
patients.columns.tolist()

['patientunitstayid',
 'patienthealthsystemstayid',
 'gender',
 'age',
 'ethnicity',
 'hospitalid',
 'wardid',
 'apacheadmissiondx',
 'admissionheight',
 'hospitaladmittime24',
 'hospitaladmitoffset',
 'hospitaladmitsource',
 'hospitaldischargeyear',
 'hospitaldischargetime24',
 'hospitaldischargeoffset',
 'hospitaldischargelocation',
 'hospitaldischargestatus',
 'unittype',
 'unitadmittime24',
 'unitadmitsource',
 'unitvisitnumber',
 'unitstaytype',
 'admissionweight',
 'dischargeweight',
 'unitdischargetime24',
 'unitdischargeoffset',
 'unitdischargelocation',
 'unitdischargestatus',
 'uniquepid']

In [24]:
import sqlite3
import pandas as pd
from config import EICU_DB  # assuming your config.py is working

# Connect to the database
conn = sqlite3.connect(EICU_DB)

In [26]:
query = "PRAGMA table_info(vitalPeriodic);"
pd.read_sql(query, conn)

,cid,name,type,notnull,dflt_value,pk
0,0,vitalperiodicid,BIGINT,0,None,0
1,1,patientunitstayid,INT,0,None,0
2,2,observationoffset,INT,0,None,0
3,3,temperature,"NUMERIC(11,4)",0,None,0
4,4,sao2,INT,0,None,0
5,5,heartrate,INT,0,None,0
6,6,respiration,INT,0,None,0
7,7,cvp,INT,0,None,0
8,8,etco2,INT,0,None,0
9,9,systemicsystolic,INT,0,None,0


In [28]:
query = """
SELECT
    p.patientunitstayid,
    p.age,
    p.gender,
    p.ethnicity,
    CASE
        WHEN p.hospitaldischargestatus = 'Expired' THEN 1
        ELSE 0
    END AS target_mortality,
    MAX(v.heartrate) AS max_heart_rate,
    MIN(v.systemicsystolic) AS min_systolic_bp,
    AVG(v.systemicdiastolic) AS avg_diastolic_bp
FROM patient p
LEFT JOIN vitalPeriodic v
    ON p.patientunitstayid = v.patientunitstayid
GROUP BY p.patientunitstayid, p.age, p.gender, p.ethnicity, target_mortality
"""

df_features = pd.read_sql(query, conn)
df_features.head()

,patientunitstayid,age,gender,ethnicity,target_mortality,max_heart_rate,min_systolic_bp,avg_diastolic_bp
0,141764,87,Female,Caucasian,0,138,,0.00000
1,141765,87,Female,Caucasian,0,138,,0.00000
2,143870,76,Male,Caucasian,0,55,53,42.71519
3,144815,34,Female,Caucasian,0,118,,0.00000
4,145427,61,Male,Caucasian,0,85,,0.00000


In [30]:
query_labs = """
SELECT
    p.patientunitstayid,
    MAX(CASE WHEN l.labname = 'WBC' THEN l.labresult END) AS max_wbc,
    MIN(CASE WHEN l.labname = 'WBC' THEN l.labresult END) AS min_wbc,
    AVG(CASE WHEN l.labname = 'WBC' THEN l.labresult END) AS avg_wbc,
    
    MAX(CASE WHEN l.labname = 'Creatinine' THEN l.labresult END) AS max_creatinine,
    MIN(CASE WHEN l.labname = 'Creatinine' THEN l.labresult END) AS min_creatinine,
    AVG(CASE WHEN l.labname = 'Creatinine' THEN l.labresult END) AS avg_creatinine,
    
    MAX(CASE WHEN l.labname = 'Sodium' THEN l.labresult END) AS max_sodium,
    MIN(CASE WHEN l.labname = 'Sodium' THEN l.labresult END) AS min_sodium,
    AVG(CASE WHEN l.labname = 'Sodium' THEN l.labresult END) AS avg_sodium
FROM patient p
LEFT JOIN lab l
    ON p.patientunitstayid = l.patientunitstayid
GROUP BY p.patientunitstayid
"""

df_labs = pd.read_sql(query_labs, conn)
df_labs.head()

,patientunitstayid,max_wbc,min_wbc,avg_wbc,max_creatinine,min_creatinine,avg_creatinine,max_sodium,min_sodium,avg_sodium
0,141764,None,None,None,None,None,None,None,None,None
1,141765,None,None,None,None,None,None,None,None,None
2,143870,None,None,None,None,None,None,None,None,None
3,144815,None,None,None,None,None,None,None,None,None
4,145427,None,None,None,None,None,None,None,None,None


In [32]:
df_all = df_features.merge(df_labs, on='patientunitstayid', how='left')
df_all.head()

,patientunitstayid,age,gender,ethnicity,target_mortality,max_heart_rate,min_systolic_bp,avg_diastolic_bp,max_wbc,min_wbc,avg_wbc,max_creatinine,min_creatinine,avg_creatinine,max_sodium,min_sodium,avg_sodium
0,141764,87,Female,Caucasian,0,138,,0.00000,None,None,None,None,None,None,None,None,None
1,141765,87,Female,Caucasian,0,138,,0.00000,None,None,None,None,None,None,None,None,None
2,143870,76,Male,Caucasian,0,55,53,42.71519,None,None,None,None,None,None,None,None,None
3,144815,34,Female,Caucasian,0,118,,0.00000,None,None,None,None,None,None,None,None,None
4,145427,61,Male,Caucasian,0,85,,0.00000,None,None,None,None,None,None,None,None,None


In [34]:
query_med = """
SELECT
    patientunitstayid,
    COUNT(DISTINCT drugname) AS num_unique_meds,
    MAX(CASE WHEN drugname LIKE '%Vasopressin%' THEN 1 ELSE 0 END) AS vasopressin,
    MAX(CASE WHEN drugname LIKE '%Norepinephrine%' THEN 1 ELSE 0 END) AS norepinephrine,
    MAX(CASE WHEN drugname LIKE '%Dopamine%' THEN 1 ELSE 0 END) AS dopamine
FROM medication
GROUP BY patientunitstayid
"""

df_med = pd.read_sql(query_med, conn)
df_med.head()

,patientunitstayid,num_unique_meds,vasopressin,norepinephrine,dopamine
0,141764,2,0,0,0
1,141765,11,0,0,0
2,143870,29,0,0,0
3,144815,3,0,0,0
4,145427,22,0,0,0


In [36]:
df_all = df_all.merge(df_med, on='patientunitstayid', how='left')
df_all.head()

,patientunitstayid,age,gender,ethnicity,target_mortality,max_heart_rate,min_systolic_bp,avg_diastolic_bp,max_wbc,min_wbc,...,max_creatinine,min_creatinine,avg_creatinine,max_sodium,min_sodium,avg_sodium,num_unique_meds,vasopressin,norepinephrine,dopamine
0,141764,87,Female,Caucasian,0,138,,0.00000,None,None,...,None,None,None,None,None,None,2.0,0.0,0.0,0.0
1,141765,87,Female,Caucasian,0,138,,0.00000,None,None,...,None,None,None,None,None,None,11.0,0.0,0.0,0.0
2,143870,76,Male,Caucasian,0,55,53,42.71519,None,None,...,None,None,None,None,None,None,29.0,0.0,0.0,0.0
3,144815,34,Female,Caucasian,0,118,,0.00000,None,None,...,None,None,None,None,None,None,3.0,0.0,0.0,0.0
4,145427,61,Male,Caucasian,0,85,,0.00000,None,None,...,None,None,None,None,None,None,22.0,0.0,0.0,0.0


In [38]:
query_apache = """
SELECT
    patientunitstayid,
    acutephysiologyscore,
    apachescore,
    predictedicumortality,
    predictedhospitalmortality
FROM apachepatientresult
"""

df_apache = pd.read_sql(query_apache, conn)
df_apache.head()

,patientunitstayid,acutephysiologyscore,apachescore,predictedicumortality,predictedhospitalmortality
0,141765,23,47,8.2471913877810981E-3,3.7319948856373762E-2
1,141765,23,47,0.01231145545549805,3.5813944311032138E-2
2,143870,43,60,1.5706796732545089E-2,0.02893216261220858
3,143870,43,60,1.7738751589020281E-2,0.02820647395637314
4,144815,25,25,1.8341987534681509E-3,4.2529613427439049E-3


In [40]:
df_all = df_all.merge(df_apache, on="patientunitstayid", how="left")

df_all.head()

,patientunitstayid,age,gender,ethnicity,target_mortality,max_heart_rate,min_systolic_bp,avg_diastolic_bp,max_wbc,min_wbc,...,min_sodium,avg_sodium,num_unique_meds,vasopressin,norepinephrine,dopamine,acutephysiologyscore,apachescore,predictedicumortality,predictedhospitalmortality
0,141764,87,Female,Caucasian,0,138,,0.00000,None,None,...,None,None,2.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN
1,141765,87,Female,Caucasian,0,138,,0.00000,None,None,...,None,None,11.0,0.0,0.0,0.0,23.0,47.0,8.2471913877810981E-3,3.7319948856373762E-2
2,141765,87,Female,Caucasian,0,138,,0.00000,None,None,...,None,None,11.0,0.0,0.0,0.0,23.0,47.0,0.01231145545549805,3.5813944311032138E-2
3,143870,76,Male,Caucasian,0,55,53,42.71519,None,None,...,None,None,29.0,0.0,0.0,0.0,43.0,60.0,1.5706796732545089E-2,0.02893216261220858
4,143870,76,Male,Caucasian,0,55,53,42.71519,None,None,...,None,None,29.0,0.0,0.0,0.0,43.0,60.0,1.7738751589020281E-2,0.02820647395637314


In [64]:
df_all = df_all.drop_duplicates(subset="patientunitstayid")

df_all.shape

(2520, 25)

In [68]:
df_all["predictedicumortality"] = pd.to_numeric(
    df_all["predictedicumortality"], errors="coerce"
)

df_all["predictedhospitalmortality"] = pd.to_numeric(
    df_all["predictedhospitalmortality"], errors="coerce"
)

In [76]:
# Fill all N/A ages and replaces over 90
df_all["age"] = df_all["age"].replace("> 89", 90)

df_all["age"] = df_all["age"].fillna(df_all["age"].median())

df_all["age"] = pd.to_numeric(df_all["age"], errors="coerce")

In [80]:
# Set Medications to 0 if not taken
med_cols = ["num_unique_meds", "vasopressin", "norepinephrine", "dopamine"]

df_all[med_cols] = df_all[med_cols].fillna(0)

In [94]:
# Replace blanks with NAN
import numpy as np

df_all = df_all.replace("", np.nan)

/var/folders/6j/f53n2jx50gg4bb1n36_wppjr0000gn/T/ipykernel_1640/4005033882.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_all = df_all.replace("", np.nan)


In [96]:
# Force columns to be numeric
cols_to_convert = [
    "avg_diastolic_bp", "min_systolic_bp", "max_heart_rate",
    "max_wbc","min_wbc","avg_wbc",
    "max_creatinine","min_creatinine","avg_creatinine",
    "max_sodium","min_sodium","avg_sodium"
]

for col in cols_to_convert:
    df_all[col] = pd.to_numeric(df_all[col], errors="coerce")

In [98]:
# Set N/A vital to median
vital_cols = ["avg_diastolic_bp", "min_systolic_bp", "max_heart_rate"]

for col in vital_cols:
    df_all[col] = df_all[col].fillna(df_all[col].median())

In [124]:
#Drop Labs as they are not complete
lab_cols = [
"max_wbc","min_wbc","avg_wbc",
"max_creatinine","min_creatinine","avg_creatinine",
"max_sodium","min_sodium","avg_sodium"
]

df_all = df_all.drop(columns=lab_cols)

In [126]:
# Score to average if N/A
apache_cols = [
    "acutephysiologyscore",
    "apachescore",
    "predictedicumortality",
    "predictedhospitalmortality"
]

for col in apache_cols:
    df_all[col] = df_all[col].fillna(df_all[col].median())

In [128]:
# Set Gender for N/A
df_all["gender"] = df_all["gender"].fillna(df_all["gender"].mode()[0])

# Set Ethnicity to Unknown
df_all["ethnicity"] = df_all["ethnicity"].fillna("Unknown")

In [130]:
df_all.isnull().sum().sort_values(ascending=False)

patientunitstayid             0
age                           0
gender                        0
ethnicity                     0
target_mortality              0
max_heart_rate                0
min_systolic_bp               0
avg_diastolic_bp              0
num_unique_meds               0
vasopressin                   0
norepinephrine                0
dopamine                      0
acutephysiologyscore          0
apachescore                   0
predictedicumortality         0
predictedhospitalmortality    0
dtype: int64

In [134]:
df_all.to_csv("../data/processed/mortality_features.csv", index=False)

In [136]:
df_all.shape

(2520, 16)

In [138]:
df_all.head()

,patientunitstayid,age,gender,ethnicity,target_mortality,max_heart_rate,min_systolic_bp,avg_diastolic_bp,num_unique_meds,vasopressin,norepinephrine,dopamine,acutephysiologyscore,apachescore,predictedicumortality,predictedhospitalmortality
0,141764,87.0,Female,Caucasian,0,138.0,78.0,0.00000,2.0,0.0,0.0,0.0,36.0,48.0,0.019762,0.040795
1,141765,87.0,Female,Caucasian,0,138.0,78.0,0.00000,11.0,0.0,0.0,0.0,23.0,47.0,0.008247,0.037320
3,143870,76.0,Male,Caucasian,0,55.0,53.0,42.71519,29.0,0.0,0.0,0.0,43.0,60.0,0.015707,0.028932
5,144815,34.0,Female,Caucasian,0,118.0,78.0,0.00000,3.0,0.0,0.0,0.0,25.0,25.0,0.001834,0.004253
7,145427,61.0,Male,Caucasian,0,85.0,78.0,0.00000,22.0,0.0,0.0,0.0,26.0,37.0,0.009514,0.021407


In [141]:
df_all["target_mortality"].value_counts()

target_mortality
0    2308
1     212
Name: count, dtype: int64

SyntaxError: invalid syntax (2950850723.py, line 1)